In [1]:
from pathlib import Path
from typing import List, Tuple, Dict
import json, random, itertools, statistics as st

import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import KFold, train_test_split
from collections import Counter
from sklearn.neighbors import BallTree
from sklearn.feature_extraction.text import CountVectorizer
from peft import LoraConfig, get_peft_model, PeftModel, PeftConfig
import transformers, importlib

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)

/home/hartb/Estudos/mestrado/ner-splits/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from transformers.generation.logits_process import TopKLogitsWarper, TopPLogitsWarper
import torch
import transformers


def top_k_top_p_filtering(
    logits: torch.Tensor,
    top_k: int = 0,
    top_p: float = 1.0,
    filter_value: float = -float("Inf"),
    min_tokens_to_keep: int = 1,
):
    """
    Reimplementação mínima usada pelo TRL 0.7.
    Aplica top-k e/ou nucleus (top-p) nos logits em lote.
    """
    if top_k > 0:
        logits = TopKLogitsWarper(top_k=top_k, min_tokens_to_keep=min_tokens_to_keep)(
            None, logits
        )
    if 0 < top_p < 1.0:
        logits = TopPLogitsWarper(top_p=top_p, min_tokens_to_keep=min_tokens_to_keep)(
            None, logits
        )
    return logits


# injeta no namespace que o TRL espera
transformers.top_k_top_p_filtering = top_k_top_p_filtering

In [3]:
from trl import SFTTrainer
from seqeval.metrics import f1_score, classification_report
from tqdm.auto import tqdm

# Configuração e Verificação Inicial

In [13]:
JSON_PATH = "data/geocorpus-v2.json"        # ajuste se estiver noutra pasta
SEED_GLOBAL = 42
BASE_MODEL = "gpt2-large"
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
random.seed(SEED_GLOBAL)
np.random.seed(SEED_GLOBAL)
torch.manual_seed(SEED_GLOBAL)

# ---------- ler o arquivo ----------
with open(JSON_PATH, encoding="utf-8") as f:
    raw = json.load(f)

# cada entrada já tem tokens + ner_tokens

In [14]:
records_geo = [
    {
        "sentence_id": i,
        "tokens"     : item["tokens"],
        "ner_tags"   : item["ner_tokens"],
    }
    for i, item in enumerate(raw)
]

geocorpus_full = Dataset.from_list(records_geo)

In [15]:
# lista de rótulos (ordem alfabética garante consistência entre runs)
label_list = sorted({l for sent in geocorpus_full["ner_tags"] for l in sent})
label2id   = {l: i for i, l in enumerate(label_list)}
id2label   = {i: l for l, i in label2id.items()}
NUM_LABELS = len(label_list)

# Splits

In [16]:
def holdout_split(dataset: Dataset, pct_test: float = 0.2, seed: int = 42):
    idx = np.arange(len(dataset))
    tr, te = train_test_split(idx, test_size=pct_test, random_state=seed)
    return {"train": dataset.select(tr), "test": dataset.select(te)}


def loc_split(dataset: Dataset, pct_test: float = 0.2, ngram: int = 4, seed: int = 42):
    docs = [" ".join(t) for t in dataset["tokens"]]
    vect = CountVectorizer(
        analyzer="word", ngram_range=(ngram, ngram), binary=True
    ).fit(docs)
    X = vect.transform(docs).astype(bool)
    inter = X @ X.T
    sz = X.sum(1).A1
    union = (sz[:, None] + sz[None, :]) - inter.A
    np.fill_diagonal(union, 1)
    jac = (inter.A / union).mean(1)  # overlap médio
    order = np.argsort(jac)  # menor → teste
    n_test = int(len(dataset) * pct_test)
    te, tr = order[:n_test], order[n_test:]
    return {"train": dataset.select(tr), "test": dataset.select(te)}

In [17]:
def len_split(
    dataset: Dataset,
    pct_test: float = 0.2,
    seed: int = 42,
    longest_as_test: bool = True,
):
    """Teste = sentenças mais longas (default) ou mais curtas."""
    lengths = np.array([len(t) for t in dataset["tokens"]])
    order = np.argsort(-lengths) if longest_as_test else np.argsort(lengths)
    n_test = int(len(dataset) * pct_test)
    te, tr = order[:n_test], order[n_test:]
    return {"train": dataset.select(tr), "test": dataset.select(te)}

In [18]:
def split_heur_length(ds, top_pct=0.20):
    lengths = np.array([len(t) for t in ds["tokens"]])
    thr = np.percentile(lengths, 100 * (1 - top_pct))
    idx_long = np.where(lengths >= thr)[0]
    idx_short = np.where(lengths < thr)[0]
    return DatasetDict(
        train=ds.select(idx_short.tolist()), dev=ds.select(idx_long.tolist())
    )

In [19]:
def rarity_score(labels_seq, freq_dict):
    # raridade = soma(1/freq) dos tipos únicos na sentença
    seen = {lab[2:] for lab in labels_seq if lab != "O"}
    return sum(1 / freq_dict[t] for t in seen) if seen else 0.0


def rarity_split(dataset: Dataset, pct_test: float = 0.2, seed: int = 42):
    # ❶ contar frequências globais por tipo
    all_types = [lab[2:] for seq in dataset["ner_tokens"] for lab in seq if lab != "O"]
    freq = Counter(all_types)
    # ❷ calcular score de cada sentença
    scores = np.array([rarity_score(seq, freq) for seq in dataset["ner_tokens"]])
    order = np.argsort(-scores)  # mais raras primeiro
    n_test = int(len(dataset) * pct_test)
    te, tr = order[:n_test], order[n_test:]
    return {"train": dataset.select(tr), "test": dataset.select(te)}

In [20]:
# ---------- 1.3 Reverse-Curriculum easy→hard ------------------------
def difficulty_scores(dataset: Dataset):
    length = np.array([len(t) for t in dataset["tokens"]])
    dens = np.array(
        [sum(l != "O" for l in labs) / len(labs) for labs in dataset["ner_tokens"]]
    )
    return (length - length.mean()) / length.std() + (dens - dens.mean()) / dens.std()


def curriculum_split(dataset: Dataset, pct_test: float = 0.2, seed: int = 42):
    s = difficulty_scores(dataset)
    order = np.argsort(s)  # easy→hard
    n_test = int(len(dataset) * pct_test)
    te, tr = order[-n_test:], order[:-n_test]
    return {"train": dataset.select(tr), "test": dataset.select(te)}

In [21]:
ACTIVE_SPLITS = {
    #"holdout": holdout_split,
    #"loc": loc_split,
    #"curriculum": curriculum_split,
    "len": len_split,  # ← heurística de tamanho
    #"rarity": rarity_split,  # ← heurística de raridade
}

# Tokenização e Métricas

In [22]:
def inline_tags(tokens, labels):
    out, open_tag = [], None
    for tok, lab in zip(tokens, labels):
        if lab.startswith("B-"):
            if open_tag:
                out.append(f"</{open_tag}>")
            open_tag = lab[2:]
            out.append(f"<{open_tag}>{tok}")
        elif lab.startswith("I-"):
            out.append(tok)
        else:
            if open_tag:
                out.append(f"</{open_tag}>")
                open_tag = None
            out.append(tok)
    if open_tag:
        out.append(f"</{open_tag}>")
    return " ".join(out)

In [27]:
def build_sft_dataset(raw_ds):
    rows = []
    for ex in raw_ds:
        sent = " ".join(ex["tokens"])
        tagged = inline_tags(ex["tokens"], ex["ner_tags"])
        prompt = "You are an NER tagger for Portuguese.\n" f"Sentence: {sent}\nTags:"
        rows.append({"text": prompt + " " + tagged})  # ← uma coluna só
    return Dataset.from_list(rows)

In [34]:
for split_name, split_fn in ACTIVE_SPLITS.items():
    print(f"\n=== Split: {split_name} ===")
    parts = split_fn(geocorpus_full)  # {'train': .. , 'test': ..}
    ds_train = build_sft_dataset(parts["train"])
    ds_test = build_sft_dataset(parts["test"])

    # ---- modelo 4-bit -------------------------------------------------
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )
    tok = AutoTokenizer.from_pretrained(BASE_MODEL)
    tok.pad_token = tok.eos_token

    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, quantization_config=bnb_cfg, device_map="auto"
    )
    lora_cfg = LoraConfig(
        r=8,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["c_attn", "c_proj"],
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(base, lora_cfg)

    # ---- treinamento --------------------------------------------------
    args = TrainingArguments(
        output_dir=str(OUTPUT_DIR / split_name),
        num_train_epochs=3,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=50,
        seed=SEED_GLOBAL,
        gradient_checkpointing=True,
    )

    trainer = SFTTrainer(
        model=model,
        args=args,
        train_dataset=ds_train,
        eval_dataset=ds_test,
        dataset_text_field="text",  # ou formatting_func=fmt
        max_seq_length=512,
    )

    trainer.train()
    trainer.save_model(str(OUTPUT_DIR / split_name))
    tok.save_pretrained(str(OUTPUT_DIR / split_name))
    print(f"✔️  Adapter salvo em {OUTPUT_DIR/split_name}\n")


=== Split: len ===


OutOfMemoryError: CUDA out of memory. Tried to allocate 2.00 MiB. GPU 0 has a total capacity of 3.81 GiB of which 704.00 KiB is free. Including non-PyTorch memory, this process has 1.53 GiB memory in use. Process 67810 has 2.28 GiB memory in use. Of the allocated memory 1.05 GiB is allocated by PyTorch, and 419.65 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)